In [39]:
import numpy as np
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from pathlib import Path
import pandas as pd

In [40]:
data_path = Path("/Users/patrykksiazek/Desktop/Machine-Learning---Preliminary-Data-Analysis/")

In [41]:
grouped_train_logs = pd.read_csv(data_path / 'grouped_train_logs_advanced.csv').drop(columns={'Unnamed: 0'})
X_train_scores = pd.read_csv(data_path / 'X_train_scores.csv').drop(columns={'Unnamed: 0'})
X_test_scores = pd.read_csv(data_path / 'X_test_scores.csv').drop(columns={'Unnamed: 0'})

In [42]:
cols_to_drop = ['id', 'max_up_time', 'min_down_time', 'activity_len_per_essay_Nonproduction']

In [43]:
# Splitting the grouped_train_logs into train and test sets based on IDs
train_data = grouped_train_logs.merge(X_train_scores, on='id').drop(columns=cols_to_drop)
test_data = grouped_train_logs.merge(X_test_scores, on='id').drop(columns=cols_to_drop)

In [44]:
# Separating features and target variable
X_train = train_data.drop(['score'], axis=1)
y_train = train_data['score']
X_test = test_data.drop(['score'], axis=1)
y_test = test_data['score']

In [45]:
param_space = {'n_estimators': hp.quniform('n_estimators', 10, 200, 1), 'max_depth': hp.quniform('max_depth', 1, 16, 1), 'min_samples_split': hp.quniform('min_samples_split', 2, 20, 1), 'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 20, 1)}

In [46]:
191*16*19*20

1161280

In [47]:
def objective(params):
    rf = RandomForestRegressor(
        n_estimators=int(params['n_estimators']),
        max_depth=int(params['max_depth']),
        min_samples_split=int(params['min_samples_split']),
        min_samples_leaf=int(params['min_samples_leaf']),
        random_state=42
    )
    score = -np.mean(cross_val_score(rf, X_train, y_train, cv=5, scoring='neg_mean_squared_error'))
    
    return {'loss': score, 'status': STATUS_OK}

In [48]:
trials = Trials()

In [49]:
best = fmin(fn=objective, space=param_space, algo=tpe.suggest, max_evals=50, trials=trials, verbose=3, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

100%|██████████| 50/50 [06:43<00:00,  8.08s/trial, best loss: 0.467418923866383]  


In [50]:
best

{'max_depth': np.float64(16.0),
 'min_samples_leaf': np.float64(13.0),
 'min_samples_split': np.float64(16.0),
 'n_estimators': np.float64(153.0)}

In [52]:
best_rf = RandomForestRegressor(
        n_estimators=int(best['n_estimators']),
        max_depth=int(best['max_depth']),
        min_samples_split=int(best['min_samples_split']),
        min_samples_leaf=int(best['min_samples_leaf']),
        random_state=42
    )

In [53]:
def train_and_evaluate(model, X_train=X_train, X_test = X_test):
    """
    Trenuje model i oblicza RMSE, na zbiorach treningowym i testowym
    """
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    return rmse_train, rmse_test
    

In [54]:
train_and_evaluate(best_rf)

(np.float64(0.5441593672497141), np.float64(0.6829801386127076))